# ANN Baseline

Baseline-ANN trainieren auf FE_v1 (ohne destinations) und evaluieren mit MAP@5 / HIT@5 wie bei LightGBM — aber mit kleinerem, stratifiziertem Sample (z.B. 300k).

## Setup

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
fe_path = SRC_PATH / "features" / "fe_v1.py"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH exists:", SRC_PATH.exists())
print("fe_path:", fe_path)
print("fe_path exists:", fe_path.exists())


PROJECT_ROOT: c:\Users\Philipp\AIBootcamp\ml_project
SRC_PATH exists: True
fe_path: c:\Users\Philipp\AIBootcamp\ml_project\src\features\fe_v1.py
fe_path exists: False


In [2]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

# Notebook liegt in /notebooks -> Project Root ist eine Ebene höher
PROJECT_ROOT = Path.cwd().parent

# src-Ordner explizit in den Python Path aufnehmen
SRC_PATH = PROJECT_ROOT / "src"

for p in [str(PROJECT_ROOT), str(SRC_PATH)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)
print("src exists:", SRC_PATH.exists())

PROJECT_ROOT: c:\Users\Philipp\AIBootcamp\ml_project
SRC_PATH: c:\Users\Philipp\AIBootcamp\ml_project\src
src exists: True


In [3]:
# src importierbar machen
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLE = PROJECT_ROOT / "data_sample"

DATA_RAW, DATA_PROCESSED


(WindowsPath('c:/Users/Philipp/AIBootcamp/ml_project/data/raw'),
 WindowsPath('c:/Users/Philipp/AIBootcamp/ml_project/data/processed'))

In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_path = DATA_PROCESSED / "df_model.parquet"
df = pd.read_parquet(df_path)

print("Loaded:", df_path)
print("Shape:", df.shape)
display(df.head(3))


Loaded: c:\Users\Philipp\AIBootcamp\ml_project\data\processed\df_model.parquet
Shape: (2988177, 173)


,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,...,d140,d141,d142,d143,d144,d145,d146,d147,d148,d149
0,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,...,-2.384553,-2.345528,-2.396591,-2.399953,-2.388116,-2.394294,-2.400667,-2.398716,-2.386585,-2.390370
1,2014-02-27 18:01:32,2,3,66,318,52078,NaN,756,0,1,...,-2.298266,-2.145362,-2.289405,-2.299516,-2.293402,-2.298682,-2.299516,-2.293223,-2.299516,-2.217007
2,2013-06-15 15:38:05,30,4,195,548,56440,NaN,1048,0,1,...,-2.269617,-2.158832,-2.273201,-2.137717,-2.237712,-2.235306,-2.273201,-2.273201,-2.273201,-2.273201


In [5]:
# -----------------------------
# Clean dtypes (FE_v1-kompatibel)
# -----------------------------
df_clean = df.copy()

# Target als int (kleiner Typ reicht)
df_clean["hotel_cluster"] = df_clean["hotel_cluster"].astype("int32")

# bool -> int (optional, aber ANN-freundlich)
bool_cols = df_clean.select_dtypes(include=["bool"]).columns
df_clean[bool_cols] = df_clean[bool_cols].astype("int8")

# float64 -> float32 (einheitlich, speichersparend)
float64_cols = df_clean.select_dtypes(include=["float64"]).columns
df_clean[float64_cols] = df_clean[float64_cols].astype("float32")

# WICHTIG: Date-Spalten NICHT droppen oder vor-konvertieren
# fe_v1 macht pd.to_datetime selbst.
date_cols = ["date_time", "srch_ci", "srch_co"]
for c in date_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].astype("string")  # NA-safe

print("Dtype counts:")
display(df_clean.dtypes.value_counts())
print("Date cols dtypes:")
display(df_clean[date_cols].dtypes)


Dtype counts:


float32    150
int64       19
string       3
int32        1
Name: count, dtype: int64

Date cols dtypes:


date_time    string[python]
srch_ci      string[python]
srch_co      string[python]
dtype: object

In [6]:
from sklearn.model_selection import train_test_split

N_SAMPLE = 300_000
df_small, _ = train_test_split(
    df_clean,
    train_size=N_SAMPLE,
    random_state=42,
    stratify=df_clean["hotel_cluster"]
)
print(df_small.shape)


(300000, 173)


## FE anwenden

In [7]:
from src.fe_v1 import make_features


X, y = make_features(df_small)

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head(3))
print(X.dtypes)


X shape: (300000, 17)
y shape: (300000,)


,site_name,posa_continent,user_location_country,user_location_region,srch_destination_id,srch_destination_type_id,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,checkin_month,length_of_stay,stay_type,is_mobile,is_package,channel,distance_missing,distance_bucket
339808,11,3,231,48,12224,6,2,1,1,8,2,short,0,0,9,True,unknown
1373819,2,3,66,337,12176,6,2,0,1,10,2,short,0,0,9,True,unknown
2246141,11,3,205,155,12269,6,2,2,1,7,2,short,0,0,5,False,mid


site_name                    int64
posa_continent               int64
user_location_country        int64
user_location_region         int64
srch_destination_id          int64
srch_destination_type_id     int64
srch_adults_cnt              int64
srch_children_cnt            int64
srch_rm_cnt                  int64
checkin_month                int64
length_of_stay               int64
stay_type                   object
is_mobile                    int64
is_package                   int64
channel                      int64
distance_missing              bool
distance_bucket             object
dtype: object


In [8]:
X = X.copy()
X["distance_missing"] = X["distance_missing"].astype("int8")


In [9]:
from sklearn.model_selection import train_test_split

SEED = 42

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(X_train.shape, X_val.shape)
print("n_classes:", y.nunique())


(240000, 17) (60000, 17)
n_classes: 100


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["stay_type", "distance_bucket"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(with_mean=False), num_cols),  # Output bleibt sparse
    ],
    remainder="drop"
)

X_train_p = preprocess.fit_transform(X_train)
X_val_p   = preprocess.transform(X_val)

print("Processed:", X_train_p.shape, X_val_p.shape, "sparse:", hasattr(X_train_p, "toarray"))


Processed: (240000, 23) (60000, 23) sparse: False


In [11]:
import numpy as np

def hit_at_k(y_true, y_prob, k=5):
    topk = np.argsort(-y_prob, axis=1)[:, :k]
    return np.mean([yt in topk[i] for i, yt in enumerate(y_true)])

def map_at_k(y_true, y_prob, k=5):
    topk = np.argsort(-y_prob, axis=1)[:, :k]
    s = 0.0
    for i, yt in enumerate(y_true):
        row = topk[i]
        if yt in row:
            rank = np.where(row == yt)[0][0] + 1
            s += 1.0 / rank
    return s / len(y_true)


## Baseline Model Training

In [12]:
import tensorflow as tf
from tensorflow.keras import layers

tf.random.set_seed(SEED)

X_train_nn = X_train_p.toarray() if hasattr(X_train_p, "toarray") else X_train_p
X_val_nn   = X_val_p.toarray()   if hasattr(X_val_p, "toarray")   else X_val_p

n_classes = int(np.max(y_train) + 1)

model = tf.keras.Sequential([
    layers.Input(shape=(X_train_nn.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(n_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)

model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               6144      
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 100)               25700     
                                                                 
Total params: 31844 (124.39 KB)
Trainable params: 31844 (124.39 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    X_train_nn, y_train,
    validation_data=(X_val_nn, y_val),
    epochs=15,
    batch_size=2048,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/15


118/118 [==============================] - 1s 7ms/step - loss: 4.3638 - sparse_categorical_accuracy: 0.0419 - val_loss: 4.2233 - val_sparse_categorical_accuracy: 0.0556 - lr: 0.0010
Epoch 2/15
118/118 [==============================] - 1s 5ms/step - loss: 4.2162 - sparse_categorical_accuracy: 0.0537 - val_loss: 4.1718 - val_sparse_categorical_accuracy: 0.0595 - lr: 0.0010
Epoch 3/15
118/118 [==============================] - 1s 5ms/step - loss: 4.1800 - sparse_categorical_accuracy: 0.0573 - val_loss: 4.1549 - val_sparse_categorical_accuracy: 0.0608 - lr: 0.0010
Epoch 4/15
118/118 [==============================] - 1s 5ms/step - loss: 4.1620 - sparse_categorical_accuracy: 0.0588 - val_loss: 4.1427 - val_sparse_categorical_accuracy: 0.0622 - lr: 0.0010
Epoch 5/15
118/118 [==============================] - 1s 5ms/step - loss: 4.1488 - sparse_categorical_accuracy: 0.0605 - val_loss: 4.1326 - val_sparse_categorical_accuracy: 0.0624 - lr: 0.0010
Epoch 6/15
118/118 [=============

## Evaluation

In [14]:
y_val_prob = model.predict(X_val_nn, batch_size=4096, verbose=0)

y_val_arr = y_val.to_numpy() if hasattr(y_val, "to_numpy") else np.asarray(y_val)

print("HIT@5:", hit_at_k(y_val_arr, y_val_prob, k=5))
print("MAP@5:", map_at_k(y_val_arr, y_val_prob, k=5))


HIT@5: 0.22798333333333334
MAP@5: 0.1213949999999917


## 1st Iteration - more capacity + batchnorm

In [15]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers

tf.random.set_seed(SEED)

model = tf.keras.Sequential([
    layers.Input(shape=(X_train_nn.shape[1],)),

    layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-5)),
    layers.BatchNormalization(),
    layers.Dropout(0.35),

    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-5)),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(n_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 512)               12288     
                                                                 
 batch_normalization (Batch  (None, 512)               2048      
 Normalization)                                                  
                                                                 
 dropout_1 (Dropout)         (None, 512)               0         
                                                                 
 dense_3 (Dense)             (None, 256)               131328    
                                                                 
 batch_normalization_1 (Bat  (None, 256)               1024      
 chNormalization)                                                
                                                                 
 dropout_2 (Dropout)         (None, 256)              

In [17]:
import tensorflow as tf

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=n_classes)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes=n_classes)



In [18]:
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4),
    loss=loss,
    metrics=["categorical_accuracy"]
)


In [19]:
history = model.fit(
    X_train_nn, y_train_oh,
    validation_data=(X_val_nn, y_val_oh),
    epochs=15,
    batch_size=2048,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/15
118/118 [==============================] - 3s 15ms/step - loss: 4.7592 - categorical_accuracy: 0.0339 - val_loss: 4.3720 - val_categorical_accuracy: 0.0522 - lr: 8.0000e-04
Epoch 2/15
118/118 [==============================] - 2s 14ms/step - loss: 4.4103 - categorical_accuracy: 0.0459 - val_loss: 4.2507 - val_categorical_accuracy: 0.0594 - lr: 8.0000e-04
Epoch 3/15
118/118 [==============================] - 2s 14ms/step - loss: 4.3061 - categorical_accuracy: 0.0516 - val_loss: 4.2063 - val_categorical_accuracy: 0.0614 - lr: 8.0000e-04
Epoch 4/15
118/118 [==============================] - 2s 14ms/step - loss: 4.2589 - categorical_accuracy: 0.0553 - val_loss: 4.1819 - val_categorical_accuracy: 0.0634 - lr: 8.0000e-04
Epoch 5/15
118/118 [==============================] - 2s 14ms/step - loss: 4.2280 - categorical_accuracy: 0.0586 - val_loss: 4.1695 - val_categorical_accuracy: 0.0636 - lr: 8.0000e-04
Epoch 6/15
118/118 [==============================] - 2s 14ms/step - loss: 4.203

In [21]:
y_val_prob = model.predict(X_val_nn, batch_size=4096, verbose=0)

y_val_arr = y_val.to_numpy() if hasattr(y_val, "to_numpy") else np.asarray(y_val)

print("HIT@5:", hit_at_k(y_val_arr, y_val_prob, k=5))
print("MAP@5:", map_at_k(y_val_arr, y_val_prob, k=5))


HIT@5: 0.24761666666666668
MAP@5: 0.13161083333332355


## Embeddings

In [22]:
from sklearn.model_selection import train_test_split

SEED = 42

X = X.copy()
X["distance_missing"] = X["distance_missing"].astype("int8")  # wichtig

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

n_classes = int(y_train.max() + 1)
print(X_train.shape, X_val.shape, "n_classes:", n_classes)


(240000, 17) (60000, 17) n_classes: 100


In [23]:
int_cat_cols = [
    "site_name",
    "posa_continent",
    "user_location_country",
    "user_location_region",
    "srch_destination_id",
    "srch_destination_type_id",
    "channel",
]

str_cat_cols = ["stay_type", "distance_bucket"]

num_cols = [
    "srch_adults_cnt",
    "srch_children_cnt",
    "srch_rm_cnt",
    "checkin_month",
    "length_of_stay",
    "is_mobile",
    "is_package",
    "distance_missing",
]

# sanity
for c in int_cat_cols + str_cat_cols + num_cols:
    assert c in X_train.columns, f"Missing column: {c}"


In [29]:
import tensorflow as tf
import numpy as np

def df_to_dict(df):
    out = {}

    # int categorical (1D)
    for c in int_cat_cols:
        out[c] = df[c].astype("int32").to_numpy()

    # string categorical (1D)
    for c in str_cat_cols:
        out[c] = df[c].astype(str).to_numpy()

    # numeric as ONE vector input called "num"
    out["num"] = df[num_cols].astype("float32").to_numpy()

    return out


BATCH = 4096

train_ds = tf.data.Dataset.from_tensor_slices((df_to_dict(X_train), y_train.astype("int32").to_numpy()))
val_ds   = tf.data.Dataset.from_tensor_slices((df_to_dict(X_val),   y_val.astype("int32").to_numpy()))

train_ds = train_ds.shuffle(200_000, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)


In [30]:
batch_x, batch_y = next(iter(train_ds))
print(batch_x.keys())
print("num shape:", batch_x["num"].shape)


dict_keys(['site_name', 'posa_continent', 'user_location_country', 'user_location_region', 'srch_destination_id', 'srch_destination_type_id', 'channel', 'stay_type', 'distance_bucket', 'num'])
num shape: (4096, 8)


In [31]:
# Integer lookups
int_lookups = {}
for c in int_cat_cols:
    layer = tf.keras.layers.IntegerLookup(output_mode="int", mask_value=-1)  # -1 als unknown
    layer.adapt(X_train[c].astype("int32").to_numpy())
    int_lookups[c] = layer

# String lookups
str_lookups = {}
for c in str_cat_cols:
    layer = tf.keras.layers.StringLookup(output_mode="int", mask_token=None)  # unknown wird automatisch gehandhabt
    layer.adapt(X_train[c].astype(str).to_numpy())
    str_lookups[c] = layer

# Normalization für numerische Features (als Vektor)
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(X_train[num_cols].astype("float32").to_numpy())


In [32]:
import math

def emb_dim(vocab_size):
    return int(min(50, round(math.sqrt(vocab_size)) + 1))

vocab_sizes = {}
for c, lk in int_lookups.items():
    vocab_sizes[c] = lk.vocabulary_size()
for c, lk in str_lookups.items():
    vocab_sizes[c] = lk.vocabulary_size()

for k,v in vocab_sizes.items():
    print(k, "vocab:", v, "dim:", emb_dim(v))


site_name vocab: 44 dim: 8
posa_continent vocab: 7 dim: 4
user_location_country vocab: 210 dim: 15
user_location_region vocab: 828 dim: 30
srch_destination_id vocab: 15231 dim: 50
srch_destination_type_id vocab: 7 dim: 4
channel vocab: 13 dim: 5
stay_type vocab: 4 dim: 3
distance_bucket vocab: 6 dim: 3


In [33]:
from tensorflow.keras import layers

inputs = {}
embeddings = []

# integer categorical -> lookup -> embedding
for c in int_cat_cols:
    inp = tf.keras.Input(shape=(1,), name=c, dtype=tf.int32)
    inputs[c] = inp
    x = int_lookups[c](inp)
    v = int_lookups[c].vocabulary_size()
    e = layers.Embedding(input_dim=v, output_dim=emb_dim(v), name=f"emb_{c}")(x)
    e = layers.Reshape((emb_dim(v),))(e)
    embeddings.append(e)

# string categorical -> lookup -> embedding
for c in str_cat_cols:
    inp = tf.keras.Input(shape=(1,), name=c, dtype=tf.string)
    inputs[c] = inp
    x = str_lookups[c](inp)
    v = str_lookups[c].vocabulary_size()
    e = layers.Embedding(input_dim=v, output_dim=emb_dim(v), name=f"emb_{c}")(x)
    e = layers.Reshape((emb_dim(v),))(e)
    embeddings.append(e)

# numeric -> normalization
num_inp = tf.keras.Input(shape=(len(num_cols),), name="num", dtype=tf.float32)
inputs["num"] = num_inp
num_x = norm(num_inp)

# concat all
x = layers.Concatenate()(embeddings + [num_x])

# dense trunk
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.30)(x)

out = layers.Dense(n_classes, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.summary()


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 site_name (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 posa_continent (InputLayer  [(None, 1)]                  0         []                            
 )                                                                                                
                                                                                                  
 user_location_country (Inp  [(None, 1)]                  0         []                            
 utLayer)                                                                                         
                                                                                            

## 2 iteration with embedded features

In [34]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/20


59/59 [==============================] - 5s 59ms/step - loss: 4.8457 - sparse_categorical_accuracy: 0.0297 - val_loss: 4.4640 - val_sparse_categorical_accuracy: 0.0471 - lr: 8.0000e-04
Epoch 2/20
59/59 [==============================] - 4s 54ms/step - loss: 4.2965 - sparse_categorical_accuracy: 0.0604 - val_loss: 4.2374 - val_sparse_categorical_accuracy: 0.0630 - lr: 8.0000e-04
Epoch 3/20
59/59 [==============================] - 4s 54ms/step - loss: 3.8429 - sparse_categorical_accuracy: 0.0902 - val_loss: 4.0664 - val_sparse_categorical_accuracy: 0.0652 - lr: 8.0000e-04
Epoch 4/20
59/59 [==============================] - 4s 58ms/step - loss: 3.5827 - sparse_categorical_accuracy: 0.1189 - val_loss: 3.9613 - val_sparse_categorical_accuracy: 0.0703 - lr: 8.0000e-04
Epoch 5/20
59/59 [==============================] - 5s 68ms/step - loss: 3.4101 - sparse_categorical_accuracy: 0.1457 - val_loss: 3.8440 - val_sparse_categorical_accuracy: 0.0938 - lr: 8.0000e-04
Epoch 6/20
59/59 [=============

In [35]:
y_val_prob = model.predict(val_ds, verbose=0)

y_val_arr = y_val.to_numpy() if hasattr(y_val, "to_numpy") else np.asarray(y_val)

print("HIT@5:", hit_at_k(y_val_arr, y_val_prob, k=5))
print("MAP@5:", map_at_k(y_val_arr, y_val_prob, k=5))


HIT@5: 0.49398333333333333
MAP@5: 0.2917075000000289


## 3. Iteration 500k

In [37]:
from sklearn.model_selection import train_test_split

SEED = 42
df_500k, _ = train_test_split(
    df_clean,
    train_size=500_000,
    random_state=SEED,
    stratify=df_clean["hotel_cluster"]
)


In [38]:
X, y = make_features(df_500k)
X["distance_missing"] = X["distance_missing"].astype("int8")


In [39]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

n_classes = int(y_train.max() + 1)


In [40]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (df_to_dict(X_train), y_train.astype("int32").to_numpy())
).shuffle(200_000, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (df_to_dict(X_val), y_val.astype("int32").to_numpy())
).batch(BATCH).prefetch(tf.data.AUTOTUNE)


In [41]:
# int_lookups adapt
for c in int_cat_cols:
    int_lookups[c].adapt(X_train[c].astype("int32").to_numpy())

# str_lookups adapt
for c in str_cat_cols:
    str_lookups[c].adapt(X_train[c].astype(str).to_numpy())

# norm adapt
norm.adapt(X_train[num_cols].astype("float32").to_numpy())


In [42]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/20
98/98 [==============================] - 7s 68ms/step - loss: 3.0737 - sparse_categorical_accuracy: 0.2013 - val_loss: 2.9547 - val_sparse_categorical_accuracy: 0.2189 - lr: 4.0000e-04
Epoch 2/20
98/98 [==============================] - 8s 70ms/step - loss: 3.0162 - sparse_categorical_accuracy: 0.2065 - val_loss: 2.9505 - val_sparse_categorical_accuracy: 0.2158 - lr: 4.0000e-04
Epoch 3/20
98/98 [==============================] - 7s 69ms/step - loss: 2.9837 - sparse_categorical_accuracy: 0.2086 - val_loss: 2.9504 - val_sparse_categorical_accuracy: 0.2129 - lr: 4.0000e-04
Epoch 4/20
98/98 [==============================] - 8s 70ms/step - loss: 2.9585 - sparse_categorical_accuracy: 0.2105 - val_loss: 2.9513 - val_sparse_categorical_accuracy: 0.2108 - lr: 4.0000e-04
Epoch 5/20
98/98 [==============================] - 8s 71ms/step - loss: 2.9408 - sparse_categorical_accuracy: 0.2123 - val_loss: 2.9515 - val_sparse_categorical_accuracy: 0.2100 - lr: 4.0000e-04
Epoch 6/20
98/98 [==

In [43]:
y_val_prob = model.predict(val_ds, verbose=0)
y_val_arr = y_val.to_numpy() if hasattr(y_val, "to_numpy") else np.asarray(y_val)

print("HIT@5:", hit_at_k(y_val_arr, y_val_prob, k=5))
print("MAP@5:", map_at_k(y_val_arr, y_val_prob, k=5))



HIT@5: 0.55329
MAP@5: 0.3320191666666517


In [58]:
from pathlib import Path

# Project root (falls noch nicht definiert)
PROJECT_ROOT = Path.cwd().parent  # Notebook liegt in /notebooks

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

save_path = MODEL_DIR / "ann_embeddings_fe_v1_500k_final"
model.save(save_path)

print("Saved:", save_path)



INFO:tensorflow:Assets written to: c:\Users\Philipp\AIBootcamp\ml_project\models\ann_embeddings_fe_v1_500k_final\assets


INFO:tensorflow:Assets written to: c:\Users\Philipp\AIBootcamp\ml_project\models\ann_embeddings_fe_v1_500k_final\assets


Saved: c:\Users\Philipp\AIBootcamp\ml_project\models\ann_embeddings_fe_v1_500k_final


## 4. Iteration mit 500k und FE_v2

In [44]:
from sklearn.model_selection import train_test_split

SEED = 42
df_500k, _ = train_test_split(
    df_clean,
    train_size=500_000,
    random_state=SEED,
    stratify=df_clean["hotel_cluster"]
)

In [45]:
from src.fe_v2 import make_features

X, y = make_features(df_500k)
X["distance_missing"] = X["distance_missing"].astype("int8")


In [48]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train shape:", X_train.shape)


X_train shape: (400000, 166)


In [50]:
import re

# destination latent features
dest_cols = [c for c in X.columns if re.fullmatch(r"d\d+", c)]

num_cols = [
    "srch_adults_cnt",
    "srch_children_cnt",
    "srch_rm_cnt",
    "checkin_month",
    "length_of_stay",
    "is_mobile",
    "is_package",
    "distance_missing",
] + dest_cols


In [51]:
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(X_train[num_cols].astype("float32").to_numpy())


In [52]:
BATCH = 4096

train_ds = tf.data.Dataset.from_tensor_slices(
    (df_to_dict(X_train), y_train.astype("int32").to_numpy())
).shuffle(200_000, seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (df_to_dict(X_val), y_val.astype("int32").to_numpy())
).batch(BATCH).prefetch(tf.data.AUTOTUNE)


In [53]:
# Integer lookups
int_lookups = {}
for c in int_cat_cols:
    lk = tf.keras.layers.IntegerLookup(output_mode="int", mask_value=-1)
    lk.adapt(X_train[c].astype("int32").to_numpy())
    int_lookups[c] = lk

# String lookups
str_lookups = {}
for c in str_cat_cols:
    lk = tf.keras.layers.StringLookup(output_mode="int", mask_token=None)
    lk.adapt(X_train[c].astype(str).to_numpy())
    str_lookups[c] = lk


In [54]:
import math
from tensorflow.keras import layers

def emb_dim(vocab):
    return int(min(50, round(math.sqrt(vocab)) + 1))

inputs = {}
embeddings = []

# int categorical embeddings
for c in int_cat_cols:
    inp = tf.keras.Input(shape=(1,), name=c, dtype=tf.int32)
    inputs[c] = inp
    x = int_lookups[c](inp)
    v = int_lookups[c].vocabulary_size()
    out_dim = 64 if c == "srch_destination_id" else emb_dim(v)
    e = layers.Embedding(v, out_dim)(x)
    e = layers.Reshape((out_dim,))(e)
    embeddings.append(e)

# string categorical embeddings
for c in str_cat_cols:
    inp = tf.keras.Input(shape=(1,), name=c, dtype=tf.string)
    inputs[c] = inp
    x = str_lookups[c](inp)
    v = str_lookups[c].vocabulary_size()
    e = layers.Embedding(v, emb_dim(v))(x)
    e = layers.Reshape((emb_dim(v),))(e)
    embeddings.append(e)

# numeric block
num_inp = tf.keras.Input(shape=(len(num_cols),), name="num", dtype=tf.float32)
inputs["num"] = num_inp
num_x = norm(num_inp)

# concat
x = layers.Concatenate()(embeddings + [num_x])
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.30)(x)

out = layers.Dense(n_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

model.summary()


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 site_name (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 posa_continent (InputLayer  [(None, 1)]                  0         []                            
 )                                                                                                
                                                                                                  
 user_location_country (Inp  [(None, 1)]                  0         []                            
 utLayer)                                                                                         
                                                                                            

In [55]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/20
98/98 [==============================] - 10s 80ms/step - loss: 4.0682 - sparse_categorical_accuracy: 0.0819 - val_loss: 3.6704 - val_sparse_categorical_accuracy: 0.1287 - lr: 8.0000e-04
Epoch 2/20
98/98 [==============================] - 9s 85ms/step - loss: 3.6128 - sparse_categorical_accuracy: 0.1123 - val_loss: 3.4612 - val_sparse_categorical_accuracy: 0.1396 - lr: 8.0000e-04
Epoch 3/20
98/98 [==============================] - 9s 88ms/step - loss: 3.4322 - sparse_categorical_accuracy: 0.1308 - val_loss: 3.3265 - val_sparse_categorical_accuracy: 0.1550 - lr: 8.0000e-04
Epoch 4/20
98/98 [==============================] - 10s 90ms/step - loss: 3.3093 - sparse_categorical_accuracy: 0.1484 - val_loss: 3.2255 - val_sparse_categorical_accuracy: 0.1678 - lr: 8.0000e-04
Epoch 5/20
98/98 [==============================] - 9s 89ms/step - loss: 3.2175 - sparse_categorical_accuracy: 0.1646 - val_loss: 3.1674 - val_sparse_categorical_accuracy: 0.1767 - lr: 8.0000e-04
Epoch 6/20
98/98 [

In [56]:
y_val_prob = model.predict(val_ds, verbose=0)
y_val_arr = y_val.to_numpy()

print("HIT@5:", hit_at_k(y_val_arr, y_val_prob, k=5))
print("MAP@5:", map_at_k(y_val_arr, y_val_prob, k=5))


HIT@5: 0.53103
MAP@5: 0.3127538333333206
